In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from sklearn.metrics.pairwise import haversine_distances
from scipy.optimize import linear_sum_assignment



In [ ]:
SRC_ROOT = Path("../data/training_patches2026-04-17T20:36/BD_MineriaAurifera_Peru-curated2025-09")
DST_ROOT = Path("../data/training_patches2026-04-17T20:36/BD_MineriaAurifera_Peru-curated2026-04-17")
USE_GEOTIFF_CENTER_FALLBACK = True

gdf_target = gpd.read_file('sampling_locations/BD_MineriaAurifera_Peru-curated2026-04-17.geojson')
gdf_target

In [ ]:
_custom_re = re.compile(r"_custom_(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)_")

def parse_lon_lat_from_name(path: Path):
    # filename stores lat, lon in that order
    m = _custom_re.search(path.name)
    if not m:
        return None
    lat = float(m.group(1))
    lon = float(m.group(2))
    return lon, lat

def geotiff_center_lon_lat(path: Path):
    with rasterio.open(path) as src:
        cx = src.width / 2.0
        cy = src.height / 2.0
        x, y = src.transform * (cx, cy)
    return float(x), float(y)

def get_target_lon_lat(gdf: gpd.GeoDataFrame):
    if gdf.geometry.name in gdf.columns and gdf.geometry.notna().any():
        g = gdf.to_crs("EPSG:4326")
        return g.geometry.x.to_numpy(), g.geometry.y.to_numpy(), g
    if {"lon", "lat"}.issubset(gdf.columns):
        g = gdf.copy()
        g = gpd.GeoDataFrame(
            g,
            geometry=gpd.points_from_xy(g["lon"], g["lat"]),
            crs="EPSG:4326",
        )
        return g["lon"].to_numpy(), g["lat"].to_numpy(), g
    raise ValueError("gdf_target must have geometry or lon/lat columns")


In [ ]:
src_paths = sorted(SRC_ROOT.glob("**/*.tif"))
if not src_paths:
    raise RuntimeError(f"No TIFF files found under {SRC_ROOT}")

rows = []
for p in src_paths:
    ll = parse_lon_lat_from_name(p)
    if ll is None and USE_GEOTIFF_CENTER_FALLBACK:
        ll = geotiff_center_lon_lat(p)
    if ll is None:
        continue
    lon, lat = ll
    rows.append(
        {
            "src_path": p,
            "rel_path": p.relative_to(SRC_ROOT),
            "lon": lon,
            "lat": lat,
        }
    )

src_df = pd.DataFrame(rows)
if src_df.empty:
    raise RuntimeError("No source points could be parsed from filename/georeference")

# -------------------------
# 1:1 nearest assignment (target -> unique source)
# -------------------------
t_lon, t_lat, gdf_target_ll = get_target_lon_lat(gdf_target)

n_tgt = len(gdf_target_ll)
n_src = len(src_df)
if n_src < n_tgt:
    raise RuntimeError(f"Need at least as many source TIFFs as targets: {n_src} < {n_tgt}")

# radians in [lat, lon] order
tgt_rad = np.radians(np.c_[t_lat, t_lon])
src_rad = np.radians(np.c_[src_df["lat"].to_numpy(), src_df["lon"].to_numpy()])

# Pairwise haversine distances (meters): shape (n_tgt, n_src)
earth_r_m = 6371000.0
D_m = haversine_distances(tgt_rad, src_rad) * earth_r_m

# Hungarian algorithm gives unique source column per target row
row_ind, col_ind = linear_sum_assignment(D_m)
# row_ind corresponds to target indices; should cover all targets when n_src >= n_tgt
if len(row_ind) != n_tgt:
    raise RuntimeError("Assignment did not cover all targets")

assigned_src_idx = np.empty(n_tgt, dtype=int)
assigned_src_idx[row_ind] = col_ind
assigned_dist_m = D_m[np.arange(n_tgt), assigned_src_idx]

matched = gdf_target_ll.copy()
matched["src_idx"] = assigned_src_idx
matched["match_dist_m"] = assigned_dist_m
matched["src_rel_path"] = src_df.iloc[assigned_src_idx]["rel_path"].astype(str).to_numpy()
matched["src_path"] = src_df.iloc[assigned_src_idx]["src_path"].astype(str).to_numpy()

# -------------------------
# Copy exactly one TIFF per target row (no dedup)
# -------------------------
for i in assigned_src_idx:
    src = Path(src_df.iloc[i]["src_path"])
    rel = Path(src_df.iloc[i]["rel_path"])
    dst = DST_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print(f"Targets: {n_tgt}")
print(f"Copied TIFFs: {n_tgt} (exactly one per target row, all unique)")
print(f"Median match distance (m): {np.median(assigned_dist_m):.3f}")
print(f"Max match distance (m): {np.max(assigned_dist_m):.3f}")
print(f"Output root: {DST_ROOT}")
